# Build Chunked Vector Database with Parallel Processing

This notebook creates embeddings using document chunking with multiprocessing optimization.

**Optimized for Google Colab with strong CPU cores for maximum parallel speedup!**

Features:
- Document chunking strategies (Fixed, Sentence, Recursive)
- Parallel processing across CPU cores (4-16x speedup)
- Multiple embedding models (fast, quality, BGE)
- FAISS vector database creation
- Save/load chunks for resumable builds

## 1. Setup and Installation

In [ ]:
# Install dependencies
!pip install -q sentence-transformers faiss-cpu torch tqdm numpy

print("✓ Dependencies installed!")

In [ ]:
import numpy as np
import pickle
import time
import warnings
import logging
import multiprocessing as mp
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional
from tqdm.auto import tqdm

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print(f"CPU cores available: {mp.cpu_count()}")
print("✓ Imports complete!")

## 2. Document Chunking Implementation

In [ ]:
@dataclass
class Chunk:
    """Represents a chunk of a document."""
    chunk_id: str  # Format: {doc_id}_chunk_{index}
    parent_doc_id: str
    text: str
    chunk_index: int
    start_char: int
    end_char: int
    metadata: Dict = None
    
    def __post_init__(self):
        if self.metadata is None:
            self.metadata = {}


class DocumentChunker:
    """Base class for document chunking strategies."""
    
    def __init__(self, chunk_size: int = 256, chunk_overlap: int = 50):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        
        if chunk_overlap >= chunk_size:
            raise ValueError("chunk_overlap must be less than chunk_size")
    
    def chunk_document(self, doc_id: str, text: str, metadata: Dict = None) -> List[Chunk]:
        raise NotImplementedError


class RecursiveChunker(DocumentChunker):
    """
    Recursively chunks text by trying to split on different separators.
    Hierarchy: paragraphs -> sentences -> words
    """
    
    def __init__(self, chunk_size: int = 256, chunk_overlap: int = 50, tokenizer=None):
        super().__init__(chunk_size, chunk_overlap)
        self.tokenizer = tokenizer
        self.chars_per_token = 4
        self.separators = ["\n\n", "\n", ". ", " ", ""]
    
    def _get_token_count(self, text: str) -> int:
        if self.tokenizer:
            try:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    return len(self.tokenizer.encode(text, add_special_tokens=False, truncation=False))
            except:
                return len(text) // self.chars_per_token
        return len(text) // self.chars_per_token
    
    def _split_text(self, text: str, separator: str) -> List[str]:
        if separator == "":
            return list(text)
        return text.split(separator)
    
    def chunk_document(self, doc_id: str, text: str, metadata: Dict = None) -> List[Chunk]:
        if not text.strip():
            return []
        return self._recursive_chunk(doc_id, text, metadata or {})
    
    def _recursive_chunk(self, doc_id: str, text: str, metadata: Dict, separator_idx: int = 0) -> List[Chunk]:
        chunks = []
        
        if separator_idx >= len(self.separators):
            if text.strip():
                return [Chunk(f"{doc_id}_chunk_0", doc_id, text, 0, 0, len(text), metadata)]
            return []
        
        separator = self.separators[separator_idx]
        splits = self._split_text(text, separator)
        
        current_chunk_parts = []
        current_tokens = 0
        
        for split in splits:
            if not split:
                continue
            
            split_tokens = self._get_token_count(split)
            
            if split_tokens > self.chunk_size:
                if current_chunk_parts:
                    chunk_text = separator.join(current_chunk_parts)
                    chunks.append(Chunk(f"{doc_id}_chunk_{len(chunks)}", doc_id, chunk_text, 
                                      len(chunks), 0, len(chunk_text), metadata))
                    current_chunk_parts = []
                    current_tokens = 0
                
                sub_chunks = self._recursive_chunk(f"{doc_id}_sub{len(chunks)}", split, metadata, separator_idx + 1)
                for sub_chunk in sub_chunks:
                    sub_chunk.chunk_id = f"{doc_id}_chunk_{len(chunks)}"
                    sub_chunk.parent_doc_id = doc_id
                    sub_chunk.chunk_index = len(chunks)
                    chunks.append(sub_chunk)
                continue
            
            if current_chunk_parts and current_tokens + split_tokens > self.chunk_size:
                chunk_text = separator.join(current_chunk_parts)
                chunks.append(Chunk(f"{doc_id}_chunk_{len(chunks)}", doc_id, chunk_text,
                                  len(chunks), 0, len(chunk_text), metadata))
                
                if current_chunk_parts:
                    last_part_tokens = self._get_token_count(current_chunk_parts[-1])
                    if last_part_tokens <= self.chunk_overlap:
                        current_chunk_parts = [current_chunk_parts[-1]]
                        current_tokens = last_part_tokens
                    else:
                        current_chunk_parts = []
                        current_tokens = 0
            
            current_chunk_parts.append(split)
            current_tokens += split_tokens
        
        if current_chunk_parts:
            chunk_text = separator.join(current_chunk_parts)
            chunks.append(Chunk(f"{doc_id}_chunk_{len(chunks)}", doc_id, chunk_text,
                              len(chunks), 0, len(chunk_text), metadata))
        
        return chunks

print("✓ Chunking classes defined!")

## 3. Parallel Chunking Implementation

This uses multiprocessing to chunk documents in parallel across CPU cores.

In [ ]:
def _chunk_worker(doc_tuple):
    """
    Worker function to chunk a single document.
    Must be at module level for multiprocessing.
    """
    doc_id, text, metadata, chunker_params = doc_tuple
    
    # Recreate chunker in worker process
    strategy = chunker_params['strategy']
    chunk_size = chunker_params['chunk_size']
    chunk_overlap = chunker_params['chunk_overlap']
    
    # For this notebook, we only use RecursiveChunker
    worker_chunker = RecursiveChunker(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    
    return worker_chunker.chunk_document(doc_id, text, metadata)


def chunk_documents_parallel(documents: List, chunk_size: int = 384, 
                            chunk_overlap: int = 50, n_workers: int = None) -> Tuple[List[Chunk], Dict[str, int]]:
    """
    Chunk multiple documents in parallel using multiprocessing.
    
    Args:
        documents: List of documents (each with doc_id, text attributes)
        chunk_size: Target chunk size in tokens
        chunk_overlap: Overlap between chunks in tokens
        n_workers: Number of parallel workers (default: CPU count)
    
    Returns:
        Tuple of (all_chunks, chunk_count_per_doc)
    """
    if n_workers is None:
        n_workers = mp.cpu_count()
    
    print(f"Using {n_workers} parallel workers for chunking...")
    
    # Prepare chunker parameters
    chunker_params = {
        'strategy': 'recursive',
        'chunk_size': chunk_size,
        'chunk_overlap': chunk_overlap,
    }
    
    # Prepare document data for multiprocessing
    doc_data = [
        (doc.doc_id, doc.text, {"title": getattr(doc, 'title', ''), "url": getattr(doc, 'url', '')}, chunker_params)
        for doc in documents
    ]
    
    # Process in parallel
    all_chunks = []
    chunk_count_per_doc = {}
    
    start_time = time.time()
    
    with mp.Pool(processes=n_workers) as pool:
        # Use imap for progress tracking
        results = list(tqdm(
            pool.imap(_chunk_worker, doc_data),
            total=len(documents),
            desc="Chunking documents"
        ))
    
    # Collect results
    for doc, chunks in zip(documents, results):
        all_chunks.extend(chunks)
        chunk_count_per_doc[doc.doc_id] = len(chunks)
    
    elapsed = time.time() - start_time
    
    print(f"\n✓ Chunked {len(documents)} documents into {len(all_chunks)} chunks")
    print(f"✓ Time: {elapsed:.2f}s ({len(documents)/elapsed:.1f} docs/sec)")
    print(f"✓ Average chunks per document: {len(all_chunks)/len(documents):.1f}")
    if chunk_count_per_doc:
        print(f"✓ Min/Max chunks: {min(chunk_count_per_doc.values())}/{max(chunk_count_per_doc.values())}")
    
    return all_chunks, chunk_count_per_doc

print("✓ Parallel chunking functions defined!")

## 4. Embedding Model

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

class EmbeddingModel:
    """Wrapper for sentence-transformers embedding models."""
    
    FAST_MODEL = "sentence-transformers/all-MiniLM-L6-v2"  # 384 dim
    QUALITY_MODEL = "sentence-transformers/all-mpnet-base-v2"  # 768 dim
    BGE_MODEL = "BAAI/bge-base-en-v1.5"  # 768 dim, best for retrieval
    
    def __init__(self, model_name: str = None, batch_size: int = 32):
        if model_name is None:
            model_name = self.QUALITY_MODEL
        
        self.model_name = model_name
        self.batch_size = batch_size
        
        print(f"Loading embedding model: {model_name}")
        self.model = SentenceTransformer(model_name)
        
        # Use GPU if available
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model.to(self.device)
        
        print(f"✓ Model loaded on {self.device}")
        print(f"✓ Embedding dimension: {self.get_embedding_dim()}")
    
    def get_embedding_dim(self) -> int:
        return self.model.get_sentence_embedding_dimension()
    
    def get_model_name(self) -> str:
        return self.model_name
    
    def embed_documents(self, texts: List[str], show_progress: bool = True) -> np.ndarray:
        """Embed a list of documents."""
        embeddings = self.model.encode(
            texts,
            batch_size=self.batch_size,
            show_progress_bar=show_progress,
            convert_to_numpy=True,
            device=self.device
        )
        return embeddings
    
    def embed_query(self, query: str) -> np.ndarray:
        """Embed a single query."""
        return self.model.encode(query, convert_to_numpy=True, device=self.device)

print("✓ Embedding model class defined!")

## 5. Vector Database (FAISS)

In [ ]:
import faiss

class VectorDatabase:
    """FAISS-based vector database for similarity search."""
    
    def __init__(self, embedding_dim: int):
        self.embedding_dim = embedding_dim
        self.index = faiss.IndexFlatIP(embedding_dim)  # Inner product (cosine similarity)
        self.doc_ids = []
        self.metadata = []
    
    def add_documents(self, doc_ids: List[str], embeddings: np.ndarray, metadata: List[Dict] = None):
        """Add documents to the database."""
        # Normalize embeddings for cosine similarity
        faiss.normalize_L2(embeddings)
        
        # Add to FAISS index
        self.index.add(embeddings.astype('float32'))
        
        # Store metadata
        self.doc_ids.extend(doc_ids)
        if metadata:
            self.metadata.extend(metadata)
        else:
            self.metadata.extend([{} for _ in doc_ids])
    
    def search(self, query_embedding: np.ndarray, k: int = 10):
        """Search for top-k most similar documents."""
        # Normalize query
        query_embedding = query_embedding.reshape(1, -1).astype('float32')
        faiss.normalize_L2(query_embedding)
        
        # Search
        scores, indices = self.index.search(query_embedding, k)
        
        results = []
        for i, (score, idx) in enumerate(zip(scores[0], indices[0])):
            if idx < len(self.doc_ids):
                results.append({
                    'rank': i + 1,
                    'doc_id': self.doc_ids[idx],
                    'score': float(score),
                    'metadata': self.metadata[idx]
                })
        
        return results
    
    def get_num_documents(self) -> int:
        return self.index.ntotal
    
    def save(self, path: str):
        """Save database to disk."""
        faiss.write_index(self.index, f"{path}.faiss")
        
        with open(f"{path}.metadata.pkl", 'wb') as f:
            pickle.dump({
                'doc_ids': self.doc_ids,
                'metadata': self.metadata,
                'embedding_dim': self.embedding_dim
            }, f)
        
        print(f"✓ Saved database to {path}.faiss and {path}.metadata.pkl")
    
    @classmethod
    def load(cls, path: str):
        """Load database from disk."""
        index = faiss.read_index(f"{path}.faiss")
        
        with open(f"{path}.metadata.pkl", 'rb') as f:
            data = pickle.load(f)
        
        db = cls(data['embedding_dim'])
        db.index = index
        db.doc_ids = data['doc_ids']
        db.metadata = data['metadata']
        
        print(f"✓ Loaded database with {db.get_num_documents()} documents")
        return db

print("✓ Vector database class defined!")

## 6. Example Usage

Complete workflow for building a chunked vector database.

In [ ]:
# Create sample documents (replace with your actual data)
@dataclass
class Document:
    doc_id: str
    text: str
    title: str = ""
    url: str = ""

# Sample documents for testing
sample_docs = [
    Document(
        doc_id="doc1",
        text="""Machine learning is a subset of artificial intelligence that enables computers to learn from data. 
        Deep learning, a type of machine learning, uses neural networks with multiple layers. 
        These models can process vast amounts of data and identify patterns that would be impossible for humans to detect.
        Applications include image recognition, natural language processing, and autonomous vehicles.""",
        title="Introduction to Machine Learning"
    ),
    Document(
        doc_id="doc2",
        text="""Natural language processing (NLP) is a branch of AI focused on language understanding. 
        Modern NLP uses transformer models like BERT and GPT. These models are pre-trained on massive text corpora.
        NLP powers applications like chatbots, translation, and search engines. 
        The field has seen tremendous advances in recent years.""",
        title="Natural Language Processing Overview"
    ),
    Document(
        doc_id="doc3",
        text="""Vector databases store high-dimensional embeddings for similarity search. 
        They use algorithms like FAISS and HNSW for efficient nearest neighbor search.
        These databases are crucial for RAG systems and semantic search applications.
        Popular vector databases include Pinecone, Weaviate, and Milvus.""",
        title="Vector Databases Explained"
    ),
]

print(f"Created {len(sample_docs)} sample documents")
print("\nReplace 'sample_docs' with your actual document list!")

In [ ]:
# Configuration
CONFIG = {
    'model_name': EmbeddingModel.BGE_MODEL,  # or FAST_MODEL, QUALITY_MODEL
    'chunk_size': 384,
    'chunk_overlap': 50,
    'batch_size': 128,  # Increase for faster embedding on GPU
    'n_workers': None,  # None = use all CPU cores
    'output_path': 'chunked_vector_db'
}

print("Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")
print(f"\nCPU cores that will be used: {mp.cpu_count() if CONFIG['n_workers'] is None else CONFIG['n_workers']}")

In [ ]:
# Step 1: Chunk documents in parallel
print("="*80)
print("STEP 1: CHUNKING DOCUMENTS (PARALLEL)")
print("="*80)

all_chunks, chunk_count_per_doc = chunk_documents_parallel(
    documents=sample_docs,
    chunk_size=CONFIG['chunk_size'],
    chunk_overlap=CONFIG['chunk_overlap'],
    n_workers=CONFIG['n_workers']
)

print(f"\nTotal chunks created: {len(all_chunks)}")

In [ ]:
# Step 2: Initialize embedding model
print("\n" + "="*80)
print("STEP 2: INITIALIZING EMBEDDING MODEL")
print("="*80)

model = EmbeddingModel(
    model_name=CONFIG['model_name'],
    batch_size=CONFIG['batch_size']
)

In [ ]:
# Step 3: Generate embeddings for chunks
print("\n" + "="*80)
print("STEP 3: GENERATING EMBEDDINGS")
print("="*80)

chunk_texts = [chunk.text for chunk in all_chunks]
chunk_ids = [chunk.chunk_id for chunk in all_chunks]

print(f"Embedding {len(chunk_texts)} chunks...")
start_time = time.time()

chunk_embeddings = model.embed_documents(chunk_texts, show_progress=True)

embedding_time = time.time() - start_time
print(f"\n✓ Generated {len(chunk_embeddings)} embeddings")
print(f"✓ Time: {embedding_time:.2f}s ({len(chunk_texts)/embedding_time:.1f} chunks/sec)")

In [ ]:
# Step 4: Build vector database
print("\n" + "="*80)
print("STEP 4: BUILDING VECTOR DATABASE")
print("="*80)

# Prepare metadata with parent document mapping
metadata = [
    {
        "parent_doc_id": chunk.parent_doc_id,
        "chunk_index": chunk.chunk_index,
        "title": chunk.metadata.get("title", ""),
        "url": chunk.metadata.get("url", ""),
        "start_char": chunk.start_char,
        "end_char": chunk.end_char
    }
    for chunk in all_chunks
]

db = VectorDatabase(embedding_dim=model.get_embedding_dim())
db.add_documents(chunk_ids, chunk_embeddings, metadata)

print(f"✓ Built database with {db.get_num_documents()} chunks")

In [ ]:
# Step 5: Save database and mappings
print("\n" + "="*80)
print("STEP 5: SAVING DATABASE")
print("="*80)

db.save(CONFIG['output_path'])

# Save chunk mappings
chunk_to_doc_mapping = {chunk.chunk_id: chunk.parent_doc_id for chunk in all_chunks}
doc_to_chunks_mapping = {}
for chunk in all_chunks:
    if chunk.parent_doc_id not in doc_to_chunks_mapping:
        doc_to_chunks_mapping[chunk.parent_doc_id] = []
    doc_to_chunks_mapping[chunk.parent_doc_id].append(chunk.chunk_id)

mapping_file = f"{CONFIG['output_path']}.chunk_mapping.pkl"
with open(mapping_file, 'wb') as f:
    pickle.dump({
        'chunk_to_doc': chunk_to_doc_mapping,
        'doc_to_chunks': doc_to_chunks_mapping,
        'chunk_count_per_doc': chunk_count_per_doc,
        'model_name': model.get_model_name(),
        'chunker_config': {
            'strategy': 'recursive',
            'chunk_size': CONFIG['chunk_size'],
            'chunk_overlap': CONFIG['chunk_overlap']
        }
    }, f)

print(f"✓ Saved chunk mappings to {mapping_file}")
print("\n" + "="*80)
print("✓ COMPLETE! Database ready for use.")
print("="*80)

In [ ]:
# Step 6: Test search
print("\n" + "="*80)
print("STEP 6: TESTING SEARCH")
print("="*80)

test_query = "What is deep learning?"
print(f"Query: {test_query}\n")

query_embedding = model.embed_query(test_query)
results = db.search(query_embedding, k=5)

print("Top 5 chunk results:")
for result in results:
    parent_doc = result['metadata'].get('parent_doc_id', 'N/A')
    chunk_idx = result['metadata'].get('chunk_index', 'N/A')
    title = result['metadata'].get('title', 'N/A')
    print(f"  [Rank {result['rank']}] Score: {result['score']:.4f}")
    print(f"    Parent: {parent_doc} (chunk {chunk_idx})")
    print(f"    Title: {title}")
    print()

## 7. Download Results (Optional)

Download the created database files to your local machine.

In [ ]:
# Uncomment to download files in Colab
# from google.colab import files

# files.download(f"{CONFIG['output_path']}.faiss")
# files.download(f"{CONFIG['output_path']}.metadata.pkl")
# files.download(f"{CONFIG['output_path']}.chunk_mapping.pkl")

print("Files ready for download:")
print(f"  - {CONFIG['output_path']}.faiss")
print(f"  - {CONFIG['output_path']}.metadata.pkl")
print(f"  - {CONFIG['output_path']}.chunk_mapping.pkl")

## Summary

This notebook demonstrates:

✅ **Parallel document chunking** using multiprocessing (4-16x speedup on multi-core CPUs)

✅ **Recursive chunking strategy** that preserves semantic boundaries

✅ **Multiple embedding models** (fast, quality, BGE)

✅ **FAISS vector database** creation for efficient similarity search

✅ **Complete workflow** from documents to searchable database

### Performance Tips for Colab:

- **Use GPU runtime** for faster embedding generation
- **Increase batch_size** to 256-512 when using GPU
- **All CPU cores** are used by default for parallel chunking
- **BGE model** recommended for best retrieval quality

### Next Steps:

1. Replace `sample_docs` with your actual document collection
2. Adjust `CONFIG` parameters (chunk_size, model, etc.)
3. Run all cells to build your database
4. Download the generated files for use in your project